# 2 · Reading documents

⏱ about 10 minutes

To answer from the papers, the bot first has to read them. A PDF only stores where to draw each
letter, not what is a heading, a paragraph or a table.
[Docling](https://github.com/docling-project/docling) looks at every page with a layout model,
works out its structure and writes it out as Markdown.

![Docling finds headings, tables and figures on a PDF page and writes Markdown](img/docling.svg)

We convert page by page and keep the file name and page number with every page; notebook 4
needs them for the citations. This takes about a minute.

In [1]:
from pathlib import Path

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

# These PDFs contain real text, so we skip OCR (reading text out of images).
converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=PdfPipelineOptions(do_ocr=False))}
)

pages = []
for pdf in sorted(Path("data").glob("*.pdf")):
    document = converter.convert(pdf).document
    for number in document.pages:
        pages.append({"source": pdf.name, "page": number, "text": document.export_to_markdown(page_no=number)})

print(len(pages), "pages")

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

34 pages


Here is page 3 of the Schmidt paper as Docling sees it. The table came through as a table.

In [2]:
from IPython.display import Markdown

page = next(p for p in pages if p["source"].startswith("Schmidt") and p["page"] == 3)
Markdown(page["text"])

Figure 1. Principle of the multiplexed aptamer binding assay using VideoScan technology. (1) Dye/sizeencoded microbead populations presenting different proteins on their surfaces are mixed (2) incubated with a fluorescence-labelled aptamer (3) dissolved in a binding buffer of choice. After removing unbound aptamers by washing, the microbead suspension was transferred into a cavity of a 96 well plate (4). The microbeads were allowed to settle down forming a microbead chip on the transparent bottom (5). A fluorescence microscope (6) is used to take pictures of the microbead chip (7). Imaging software analyses the pictures, recognises and counts microbeads, measures their surface fluorescence intensity and groups them into populations (8). Finally, for each population the referenced mean fluorescence intensity (rMFI) per microbead population is calculated (9). (10) Shows a 96-well microtiter plate placed into the VideoScan system.

<!-- image -->

aptamers are used in combination. By performing analyses in a 96-well format and multiplexing microbeads, we were able to sample large quantities efficiently and in a cost effective manner.

## Results and discussion

Aptamer-target binding using a multiplex assay. Our experimental approach presented in Fig. 1 required dye/size-encoded microbead populations presenting potential aptamer target proteins and some nontarget proteins (as specificity controls) on their surfaces. We coupled different proteins via EDC-chemistry to carboxylated microbead populations and checked for successful coupling with appropriate detection probes (Supplementary Table S2). Results are shown in supplementary information (Supplementary Fig. S1).

Due to non-directional binding, the target proteins have random orientation on the surface. The orientation may have an influence on the binding (e.g., inaccessible binding site, deformation) or activity (e.g., catalysed activity) 13,22 . But we argue that the random orientation can also stochastically make accessible a portion that is usually sufficient for the generation of a measurement signal. This is also in line with our previous ­work 23-25 , where molecules were successfully bound.

We then examined the binding of the nine aptamers to microbeads presenting their target molecules (Fig. 2A). We were unable to detect any signal mIgG, although the presence of the corresponding target coupled to the surface of the microbeads was clearly shown by the use of antibodies (Supplementary Fig. S1). Aptamers are considered as a substitute for antibodies, and researchers often assume that aptamers can be modified in a similar

͹

Vol.:(0123456789)

## Chunks

A whole page is too much text to hand the model for every question, and too coarse to find the
one paragraph that matters. So we cut every page into **chunks** of about 1000 characters.
Neighbouring chunks overlap by 200 characters, so a sentence cut at a border still appears whole
in one of them.

![A page cut into three overlapping chunks](img/chunking.svg)

In [3]:
def chunk(text, size=1000, overlap=200):
    starts = range(0, max(len(text) - overlap, 1), size - overlap)
    return [text[start : start + size] for start in starts]


chunks = [
    {"source": p["source"], "page": p["page"], "text": piece}
    for p in pages
    for piece in chunk(p["text"])
    if piece.strip()
]
print(len(chunks), "chunks\n")
print(chunks[10]["text"])

197 chunks

om temperature (20 °C). The samples are continuously stirred during spectra and decay curve recording to prevent precipitation of beads. For steady state measurements, a xenon lamp has been used as excitation light source. Pulsed excitation for time-resolved measurements was provided by a Fianium Supercontinuum SC400-2-PP (NKT Photonics A/S) laser operated at a repetition rate of 10 MHz (for organic dyes) or 5 MHz (for QD-beads). In time-correlated single photon counting (TCSPC) experiments with an R38090U-50 (Hamamatsu Photonics K.K.) MCP-PMT, a pulse width of the instrument response function (IRF) of about 250 ps at 10 MHz was achieved.

Flow cytometer design. A new lifetime flow cytometer (LT-FCM) including a modulated laser source, novel single photon signal processing lifetime analysis electronics and signal analysis and instrument control software (CyPAD pantau) was designed (Quantum P/pantau, Quantum Analysis GmbH). The new signal processing for FCM is capable of sin

> **Look closely.** Where does this chunk start and end? Docling knows where every heading and
> table is, but our chunker ignores that and cuts every 800 characters, through the middle of a
> table if one is in the way. Better chunkers cut along the document's structure.

Notebook 3 continues with these chunks, so we save them.

In [4]:
import json

Path("chunks.json").write_text(json.dumps(chunks))
print("saved", len(chunks), "chunks to chunks.json")

saved 197 chunks to chunks.json


## A document inspector in Chainlit

This bot reads any PDF you attach to a message. Attached files arrive in `message.elements`.
Docling needs a few seconds per document; `cl.make_async` runs it in the background so the chat
stays responsive, and `cl.Step` shows it as a step you can unfold.

In [5]:
%%writefile app.py
import chainlit as cl
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

# These PDFs contain real text, so we skip OCR (reading text out of images).
converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=PdfPipelineOptions(do_ocr=False))}
)


def chunk(text, size=1000, overlap=200):
    starts = range(0, max(len(text) - overlap, 1), size - overlap)
    return [text[start : start + size] for start in starts]


@cl.on_chat_start
async def on_chat_start():
    await cl.Message(content="Attach a PDF (paperclip icon, or drag it in) and I'll show you what Docling reads.").send()


@cl.on_message
async def on_message(message: cl.Message):
    pdfs = [file for file in message.elements if file.name.lower().endswith(".pdf")]
    if not pdfs:
        await cl.Message(content="Please attach a PDF.").send()
        return

    for pdf in pdfs:
        async with cl.Step(name=f"Docling reads {pdf.name}"):
            document = (await cl.make_async(converter.convert)(pdf.path)).document
        pages = [document.export_to_markdown(page_no=number) for number in document.pages]
        chunks = [piece for text in pages for piece in chunk(text) if piece.strip()]

        await cl.Message(
            content=f"**{pdf.name}**: {len(pages)} pages, {len(chunks)} chunks. This is page 1:\n\n---\n\n{pages[0]}"
        ).send()
        # TODO 1: send the first three chunks, each as its own message.
        # TODO 2: send every table Docling found.

Overwriting app.py


In [6]:
import socket
import subprocess
import sys

# Start the chatbot, unless it's still running from an earlier notebook.
with socket.socket() as probe:
    running = probe.connect_ex(("127.0.0.1", 8000)) == 0
if not running:
    subprocess.Popen(
        [sys.executable, "-m", "chainlit", "run", "app.py", "--watch", "--headless"],
        stdout=open("chainlit.log", "w"),
        stderr=subprocess.STDOUT,
    )
print("Chatbot: http://localhost:8000")

Chatbot: http://localhost:8000


## Your turn

Edit the `%%writefile app.py` cell above, run it again, and try your change in the browser.

Open **http://localhost:8000** and attach a PDF: one from the `data/` folder, or one of your own.

1. **Where are the cuts?** Send the first three chunks, each as its own message. Then try
   `size=300` and `size=3000` in `chunk()`. How does the number of chunks change? Which size
   would you want for answering questions?
2. **Only the tables.** `document.tables` is the list of tables Docling found, and
   `table.export_to_markdown(doc=document)` turns one into Markdown. Send each table as its own
   message. The Schmidt paper has several.
3. **Bonus: side by side.** Attach the original PDF to the reply, so you can compare it with
   Docling's Markdown: pass `elements=[cl.Pdf(name=pdf.name, path=pdf.path, display="side")]`
   to `cl.Message`. Chainlit turns the element's name into a link wherever it appears in the text.

<details>
<summary><b>Solutions</b> (try first!)</summary>

**1**
```python
        for piece in chunks[:3]:
            await cl.Message(content=piece).send()
```

**2**
```python
        for table in document.tables:
            await cl.Message(content=table.export_to_markdown(doc=document)).send()
```

**3**
```python
        await cl.Message(
            content=f"**{len(pages)} pages** read from {pdf.name}",
            elements=[cl.Pdf(name=pdf.name, path=pdf.path, display="side")],
        ).send()
```
</details>